# Road Following — Interactive Data Collection & Labeling Tool

Collect and annotate lane following data by **clicking directly on the camera image** or adjusting Target X/Y sliders with **ROS Camera Topic** (`/csi_cam_0/image_raw`).

| Step | Description |
|------|-------------|
| 1 | Setup Environment & ROS Node |
| 2 | Configure Task & Dataset (Loads Existing Saved Samples) |
| 3 | Interactive Data Collection UI (Click Image OR Adjust Sliders + Save Button) |


### 1. Setup Environment & ROS Node


In [ ]:
import os
import sys
import cv2
import numpy as np
from pathlib import Path

# Add parent directory to sys.path
pass # parent_dir = Path.cwd().parent
if str(parent_dir) not in sys.path:
    pass

import rospy
from sensor_msgs.msg import Image as ROSImage
from jetracer_ai.utils import preprocess_onnx, bgr8_to_jpeg

# Initialize ROS Node
try:
    rospy.init_node('interactive_regression_notebook', anonymous=True, disable_signals=True)
    print("[+] ROS Node initialized successfully!")
except Exception as e:
    print(f"[*] ROS Node notice: {e}")


### 2. Configure Task & Dataset (Auto-Loads Existing Saved Samples)


In [ ]:
from jetracer_ai.utils import XYDataset

TASK = 'road_following'
CATEGORIES = ['apex']
DATASETS = ['A', 'B']

datasets = {}
for name in DATASETS:
    ds = XYDataset('datasets/lane_following/' + TASK + '_' + name, CATEGORIES, random_hflip=True)
    ds.refresh()
    datasets[name] = ds

dataset_name = DATASETS[0]
dataset = datasets[dataset_name]

print(f"[*] Task: {TASK}")
print(f"[*] Categories: {CATEGORIES}")
print(f"[*] Active Dataset: {dataset_name} ({dataset.get_count('apex')} existing samples loaded from disk)")


### 3. Data Collection (Click Image OR Adjust Sliders + Save Button)

**Click on the left camera image** (or adjust Target X/Y sliders and click **Save Sample**) to save a labeled data sample with (x, y) coordinates.
A green circle will appear on the right snapshot to confirm the labeled point.


In [ ]:
import ipywidgets
from IPython.display import display
import base64
import threading

latest_ros_image = None
_lock = threading.Lock()

# Unregister previous ROS subscriber if active
if 'ros_sub' in globals() and ros_sub is not None:
    try:
        ros_sub.unregister()
    except Exception:
        pass

# Interactive Sliders & Save Button (Target X/Y range 0..224 matches standard JetRacer 224x224 image)
target_x_slider = ipywidgets.IntSlider(min=0, max=224, value=112, description='Target X', layout=ipywidgets.Layout(width='320px'))
target_y_slider = ipywidgets.IntSlider(min=0, max=224, value=180, description='Target Y', layout=ipywidgets.Layout(width='320px'))
save_button     = ipywidgets.Button(description='Save Sample', button_style='success', icon='camera', layout=ipywidgets.Layout(width='160px'))

# Image Widgets
camera_html_widget = ipywidgets.HTML(
    value="<p><b>Waiting for ROS Camera Feed...</b></p>",
    layout=ipywidgets.Layout(width='230px', height='230px')
)
snapshot_widget = ipywidgets.Image(format='jpeg', width=224, height=224)

# Refresh active dataset counts from disk
dataset.refresh()

# Selectors & Counters
dataset_widget  = ipywidgets.Dropdown(options=DATASETS, description='dataset')
category_widget = ipywidgets.Dropdown(options=CATEGORIES, description='category')
count_widget    = ipywidgets.IntText(description='count', value=dataset.get_count(category_widget.value))

def update_counts(change):
    dataset.refresh()
    count_widget.value = dataset.get_count(category_widget.value)

def set_dataset(change):
    global dataset
    dataset = datasets[change['new']]
    dataset.refresh()
    count_widget.value = dataset.get_count(category_widget.value)

dataset_widget.observe(set_dataset, names='value')
category_widget.observe(update_counts, names='value')

def save_clicked_sample(x, y):
    global latest_ros_image, dataset
    if latest_ros_image is not None:
        x = int(max(0, min(224, x)))
        y = int(max(0, min(224, y)))
        
        target_x_slider.value = x
        target_y_slider.value = y
        
        # Save labeled entry (Appends new file to disk, does NOT delete existing files)
        dataset.save_entry(category_widget.value, latest_ros_image, x, y)
        
        # Create confirmation snapshot image with green circle
        snapshot = latest_ros_image.copy()
        cv2.circle(snapshot, (x, y), 8, (0, 255, 0), -1)
        cv2.putText(snapshot, f"Saved #{dataset.get_count(category_widget.value)} (X:{x}, Y:{y})", (10, 25),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 2)
        
        snapshot_widget.value = bgr8_to_jpeg(snapshot)
        count_widget.value = dataset.get_count(category_widget.value)
        print(f"[+] Appended Sample #{count_widget.value} -> (X={x}, Y={y})")

def on_save_button_clicked(b):
    save_clicked_sample(target_x_slider.value, target_y_slider.value)

save_button.on_click(on_save_button_clicked)
sys.modules['__main__'].save_clicked_sample = save_clicked_sample

def ros_image_to_cv2(msg):
    im = np.frombuffer(msg.data, dtype=np.uint8).reshape(msg.height, msg.width, -1)
    if msg.encoding in ['rgb8', 'rgb8']:
        im = cv2.cvtColor(im, cv2.COLOR_RGB2BGR)
    elif msg.encoding == 'rgba8':
        im = cv2.cvtColor(im, cv2.COLOR_RGBA2BGR)
    elif msg.encoding == 'bgra8':
        im = cv2.cvtColor(im, cv2.COLOR_BGRA2BGR)
    
    # Standardize to 224x224 so Target X/Y (0..224) spans 100% of the image frame!
    if im.shape[0] != 224 or im.shape[1] != 224:
        im = cv2.resize(im, (224, 224))
    return im

def camera_callback(msg):
    global latest_ros_image
    if not _lock.acquire(blocking=False):
        return
    try:
        latest_ros_image = ros_image_to_cv2(msg)
        
        # Draw live green target dot based on slider values!
        tx = target_x_slider.value
        ty = target_y_slider.value
        
        preview = latest_ros_image.copy()
        cv2.circle(preview, (tx, ty), 8, (0, 255, 0), -1)
        
        jpeg_bytes = bgr8_to_jpeg(preview)
        b64 = base64.b64encode(jpeg_bytes).decode('utf-8')
        
        html_str = f'''
        <div style="display: inline-block;">
            <img src="data:image/jpeg;base64,{b64}" 
                 style="width:224px; height:224px; border:2px solid #00ff00; border-radius:4px; display:block; cursor:crosshair;" 
                 onclick="if(window.onCamClick) window.onCamClick(event, this);" />
        </div>
        <script>
        if (!window.onCamClick) {{
            window.onCamClick = function(event, elem) {{
                var rect = elem.getBoundingClientRect();
                var x = Math.round((event.clientX - rect.left) * (224 / rect.width));
                var y = Math.round((event.clientY - rect.top) * (224 / rect.height));
                if (window.Jupyter && Jupyter.notebook && Jupyter.notebook.kernel) {{
                    Jupyter.notebook.kernel.execute('save_clicked_sample(' + x + ', ' + y + ')');
                }}
            }};
        }}
        </script>
        '''
        camera_html_widget.value = html_str
    except Exception as e:
        pass
    finally:
        try:
            _lock.release()
        except RuntimeError:
            pass

topic_name = "/csi_cam_0/image_raw"
ros_sub = rospy.Subscriber(topic_name, ROSImage, camera_callback, queue_size=1, buff_size=2**24)

data_collection_widget = ipywidgets.VBox([
    ipywidgets.HBox([camera_html_widget, snapshot_widget]),
    dataset_widget,
    category_widget,
    count_widget,
    target_x_slider,
    target_y_slider,
    save_button
])

display(data_collection_widget)
print(f"[*] Subscribed to ROS Camera Topic: {topic_name}")
